# Quadratic Discriminant Analysis (QDA)

Quadratic Discriminant Analysis is a classification method similar to LDA but relaxes the assumption of equal covariance matrices across classes. This allows for quadratic decision boundaries, making it more flexible for non-linearly separable data.

## Key Concepts:

- **Quadratic Decision Boundaries**: Creates curved (quadratic) decision boundaries between classes
- **Class-Specific Covariance**: Each class has its own covariance matrix
- **More Flexible**: Can model more complex class distributions than LDA
- **Bayes Classifier**: Generalization of LDA with class-specific covariance matrices

## When to Use:

- When classes have different covariance structures
- When decision boundaries are non-linear
- When you have enough training data to estimate class-specific covariances
- When LDA's linear assumptions are too restrictive

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_blobs
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Dataset 1: Synthetic Dataset with Different Covariances

Create a dataset where classes have different covariance structures to showcase QDA's strength.

In [ ]:
# Create synthetic dataset with different covariances
np.random.seed(42)

# Class 0: Compact cluster
class0 = np.random.multivariate_normal(
    mean=[0, 0],
    cov=[[1, 0.2], [0.2, 1]],
    size=300
)

# Class 1: Elongated cluster
class1 = np.random.multivariate_normal(
    mean=[3, 3],
    cov=[[2, 1.5], [1.5, 2]],
    size=300
)

# Class 2: Different orientation
class2 = np.random.multivariate_normal(
    mean=[-2, 3],
    cov=[[1, -0.8], [-0.8, 1]],
    size=300
)

X = np.vstack([class0, class1, class2])
y = np.hstack([np.zeros(300), np.ones(300), np.full(300, 2)])

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {np.bincount(y.astype(int))}")

In [ ]:
# Visualize the dataset
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Synthetic Dataset with Different Covariances')
plt.colorbar(scatter, label='Class')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {X_train_scaled.shape[0]}")
print(f"Test samples: {X_test_scaled.shape[0]}")

## Train QDA Model

In [ ]:
# Initialize and train QDA
qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train_scaled, y_train)

# Make predictions
y_pred = qda.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

## Model Evaluation

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - QDA')
plt.tight_layout()
plt.show()

## Decision Boundary Visualization

Visualize the quadratic decision boundaries created by QDA.

In [ ]:
# Create mesh grid for decision boundary
h = 0.02
x_min, x_max = X_train_scaled[:, 0].min() - 1, X_train_scaled[:, 0].max() + 1
y_min, y_max = X_train_scaled[:, 1].min() - 1, X_train_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predict on mesh grid
Z = qda.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision boundary
plt.figure(figsize=(12, 8))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
scatter = plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap='viridis', edgecolors='k')
plt.xlabel('Feature 1 (scaled)')
plt.ylabel('Feature 2 (scaled)')
plt.title('QDA Decision Boundary (Quadratic)')
plt.colorbar(scatter, label='Class')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Comparison: QDA vs LDA

Compare QDA with LDA on the same dataset to see the difference in decision boundaries.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Train LDA for comparison
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_scaled, y_train)
y_pred_lda = lda.predict(X_test_scaled)
accuracy_lda = accuracy_score(y_test, y_pred_lda)

print(f"QDA Accuracy: {accuracy:.4f}")
print(f"LDA Accuracy: {accuracy_lda:.4f}")

In [ ]:
# Visualize both decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# QDA decision boundary
Z_qda = qda.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contourf(xx, yy, Z_qda, alpha=0.3, cmap='viridis')
scatter = axes[0].scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap='viridis', edgecolors='k')
axes[0].set_xlabel('Feature 1 (scaled)')
axes[0].set_ylabel('Feature 2 (scaled)')
axes[0].set_title(f'QDA Decision Boundary (Acc: {accuracy:.4f})')
axes[0].grid(True, alpha=0.3)

# LDA decision boundary
Z_lda = lda.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z_lda, alpha=0.3, cmap='viridis')
axes[1].scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap='viridis', edgecolors='k')
axes[1].set_xlabel('Feature 1 (scaled)')
axes[1].set_ylabel('Feature 2 (scaled)')
axes[1].set_title(f'LDA Decision Boundary (Acc: {accuracy_lda:.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Dataset 2: More Complex Synthetic Data

In [ ]:
# Create a more complex dataset
X_complex, y_complex = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=4,
    n_redundant=1,
    n_classes=3,
    n_clusters_per_class=2,
    flip_y=0.1,
    random_state=42
)

print(f"Complex dataset shape: {X_complex.shape}")
print(f"Class distribution: {np.bincount(y_complex)}")

In [ ]:
# Split and scale
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_complex, y_complex, test_size=0.3, random_state=42, stratify=y_complex
)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

# Train QDA
qda_c = QuadraticDiscriminantAnalysis()
qda_c.fit(X_train_c_scaled, y_train_c)

# Predict and evaluate
y_pred_c = qda_c.predict(X_test_c_scaled)
accuracy_c = accuracy_score(y_test_c, y_pred_c)

print(f"Accuracy on complex dataset: {accuracy_c:.4f}")

## Cross-Validation

In [ ]:
# Perform cross-validation
cv_scores = cross_val_score(qda, X_train_scaled, y_train, cv=5)

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## Regularization Parameter

QDA can use regularization to handle singular covariance matrices.

In [ ]:
# Test different regularization parameters
reg_params = [0.0, 0.1, 0.5, 1.0]

for reg_param in reg_params:
    qda_reg = QuadraticDiscriminantAnalysis(reg_param=reg_param)
    qda_reg.fit(X_train_scaled, y_train)
    y_pred_reg = qda_reg.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred_reg)
    print(f"Reg param: {reg_param:.1f}, Accuracy: {acc:.4f}")

## Predict on New Data

In [ ]:
# Function to predict new samples
def predict_class(sample):
    sample_scaled = scaler.transform([sample])
    prediction = qda.predict(sample_scaled)[0]
    probabilities = qda.predict_proba(sample_scaled)[0]
    
    return int(prediction), probabilities

# Test with new samples
new_samples = [
    [0, 0],    # Near class 0
    [3, 3],    # Near class 1
    [-2, 3],   # Near class 2
    [1.5, 1.5] # Between classes
]

for sample in new_samples:
    pred_class, probs = predict_class(sample)
    print(f"\nSample: {sample}")
    print(f"Predicted class: {pred_class}")
    print("Probabilities:")
    for i, prob in enumerate(probs):
        print(f"  Class {i}: {prob:.4f}")

## Summary

### Key Takeaways:

1. **Quadratic Decision Boundaries**: QDA creates curved decision boundaries, more flexible than LDA
2. **Class-Specific Covariance**: Each class has its own covariance matrix
3. **More Parameters**: Estimates more parameters than LDA (requires more data)
4. **Better for Heterogeneous Data**: Performs better when classes have different covariance structures

### Advantages:
- More flexible than LDA
- Can model non-linear decision boundaries
- Handles classes with different covariance structures
- Better performance on complex class distributions

### Limitations:
- Requires more training data to estimate class-specific covariances
- More parameters to estimate (can overfit with small datasets)
- Computationally more expensive than LDA
- Can suffer from singular covariance matrices (needs regularization)
- May overfit if classes have similar covariances

### QDA vs LDA:

- **Use LDA when**: Classes have similar covariance structures, limited data, need simplicity
- **Use QDA when**: Classes have different covariances, sufficient data, need flexibility